# DCF Valuation Engine v2 — Forward & Reverse (Two-Stage Fading Growth)

One discounted-cash-flow engine, queried in two directions. **Forward:** project unlevered free cash flow under a growth path that starts at an initial rate and **fades linearly to the terminal rate** over a 10-year horizon (no cliff at year 5), with EBIT margin converging to its long-run historical level, a CAPM-based WACC using a **beta estimated by OLS regression of weekly returns against SPY**, dual terminal values (Gordon growth and exit multiple), and a WACC × terminal-growth sensitivity grid. **Reverse:** hold the market price fixed and solve — with a bracketed root-finder — for the growth path the market is implicitly pricing in, reported both as the implied *initial* growth and as the implied *horizon CAGR* (the number directly comparable to the company's historical CAGR, which is itself estimated by log-linear regression with the sample window labeled). The notebook validates itself: a built-in test feeds the forward output back through the reverse solver and confirms it recovers the input growth, and numerically checks that value is monotonic in growth over the solver bracket (the property that guarantees the implied growth is unique). Data: Yahoo Finance fundamentals with a manual fallback, FRED for the risk-free rate.

In [ ]:
# ============================================================================
# Cell 2 — Installs & imports
# ============================================================================
import sys, subprocess

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("yfinance", "fredapi", "pandas-datareader")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from dataclasses import dataclass
from typing import Optional, Dict, Tuple

from scipy import optimize

import yfinance as yf

try:
    from fredapi import Fred
    _HAS_FREDAPI = True
except Exception:
    _HAS_FREDAPI = False

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})

print("Imports ready. yfinance:", yf.__version__, "| fredapi available:", _HAS_FREDAPI)

## 1. Configuration

Everything keys off `CONFIG`. The two structural knobs new in v2: `PROJECTION_YEARS` is now 10 (a longer explicit horizon shifts value out of the terminal period), and `BASE_INITIAL_GROWTH` is the **year-1** growth rate — it fades linearly to `TERMINAL_GROWTH` by the final explicit year, so there is no growth cliff at the handoff to the terminal value. Any driver left as `None` is estimated from the company's own history.

In [ ]:
# ============================================================================
# Cell 4 — Configuration
# ============================================================================
CONFIG = {
    "TICKER": "AAPL",            # <-- change me

    # --- Optional FRED key (https://fred.stlouisfed.org/docs/api/api_key.html)
    "FRED_API_KEY": "",          # leave "" to use fallbacks

    # --- Projection structure (v2: fading growth over a longer horizon) ---
    "PROJECTION_YEARS": 10,
    "BASE_INITIAL_GROWTH": 0.08,   # year-1 revenue growth; fades to terminal by year N
    "TERMINAL_GROWTH": 0.025,      # perpetual growth in Gordon terminal value
    "EXIT_EV_EBITDA": 14.0,        # exit multiple for the alternative TV method

    # --- Operating drivers (None -> estimated from the company's history) ---
    "EBIT_MARGIN": None,             # starting operating margin
    "LONG_RUN_EBIT_MARGIN": None,    # margin the path converges to (None -> hist. average)
    "TAX_RATE": None,
    "DA_PCT_REV": None,
    "CAPEX_PCT_REV": None,
    "NWC_INCREMENTAL_PCT": None,     # dNWC per $ of dRevenue (None -> from history)

    # --- Beta (v2: estimated via OLS of weekly returns vs SPY) ---
    "BETA_OVERRIDE": None,           # set a number to force it
    "BETA_LOOKBACK_YEARS": 5,

    # --- WACC inputs / fallbacks ---
    "RISK_FREE_FALLBACK": 0.043,
    "EQUITY_RISK_PREMIUM": 0.050,    # Damodaran-style mature-market ERP
    "PRETAX_COST_OF_DEBT": 0.050,
    "MIN_WACC": 0.06,
    "MAX_WACC": 0.18,

    # --- Scenario deltas on the INITIAL growth rate ---
    "BULL_DELTA": 0.04,
    "BEAR_DELTA": 0.04,

    # --- Reverse-DCF solver bounds (on the implied INITIAL growth) ---
    "IMPLIED_GROWTH_MIN": -0.20,
    "IMPLIED_GROWTH_MAX": 0.80,
}

print(f"Target: {CONFIG['TICKER']} | horizon: {CONFIG['PROJECTION_YEARS']}y, "
      f"growth fades {CONFIG['BASE_INITIAL_GROWTH']:.0%} -> {CONFIG['TERMINAL_GROWTH']:.1%}")

## 2. Macro inputs (FRED, with fallback)

Risk-free rate from the 10-Year Treasury (`DGS10`), tried through two access paths before falling back to a documented constant, so a network hiccup never kills the run.

In [ ]:
# ============================================================================
# Cell 6 — Risk-free rate (FRED -> fallback)
# ============================================================================
def get_risk_free_rate(cfg) -> Tuple[float, str]:
    """Return (risk_free_decimal, source_label). Never raises."""
    key = cfg.get("FRED_API_KEY", "").strip()

    if key and _HAS_FREDAPI:
        try:
            fred = Fred(api_key=key)
            s = fred.get_series("DGS10").dropna()
            if len(s):
                return float(s.iloc[-1]) / 100.0, "FRED DGS10 (fredapi)"
        except Exception as e:
            print("  fredapi path failed:", e)

    try:
        import pandas_datareader.data as web
        from datetime import datetime, timedelta
        start = datetime.today() - timedelta(days=30)
        df = web.DataReader("DGS10", "fred", start)
        s = df["DGS10"].dropna()
        if len(s):
            return float(s.iloc[-1]) / 100.0, "FRED DGS10 (datareader)"
    except Exception as e:
        print("  datareader path failed:", e)

    return float(cfg["RISK_FREE_FALLBACK"]), "fallback constant"

risk_free, rf_source = get_risk_free_rate(CONFIG)
print(f"Risk-free rate: {risk_free:.3%}  (source: {rf_source})")

## 3. Company fundamentals (Yahoo Finance, with manual fallback)

Same defensive pull as before (alias-tolerant row lookup, `NaN` instead of crashes, data-quality gate), with one v2 addition: it now keeps **date-aligned histories** — revenue, per-year EBIT margin, and net working capital — because the v2 drivers (long-run margin, incremental NWC, regression CAGR) are estimated from those series rather than from single latest values.

In [ ]:
# ============================================================================
# Cell 8 — Pull fundamentals
# ============================================================================
def _row(df: pd.DataFrame, *aliases) -> pd.Series:
    """Find the first matching row (case-insensitive, alias-tolerant)."""
    if df is None or df.empty:
        return pd.Series(dtype="float64")
    idx_lower = {str(i).lower().strip(): i for i in df.index}
    for a in aliases:
        k = a.lower().strip()
        if k in idx_lower:
            return df.loc[idx_lower[k]].astype("float64")
    for a in aliases:  # loose contains-match
        for low, orig in idx_lower.items():
            if a.lower().strip() in low:
                return df.loc[orig].astype("float64")
    return pd.Series(dtype="float64")

def _latest(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if not len(s):
        return np.nan
    try:
        s = s.sort_index()   # robust to ordering; latest = last after sort
    except Exception:
        pass
    return float(s.iloc[-1])

def _hist(series: pd.Series) -> pd.Series:
    """Clean numeric history, oldest -> newest."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    try:
        s = s.sort_index()
    except Exception:
        pass
    return s

def fetch_fundamentals(ticker: str) -> Dict:
    tk = yf.Ticker(ticker)
    out = {"ticker": ticker, "ok": True, "warnings": []}

    inc = getattr(tk, "income_stmt", pd.DataFrame())
    bs  = getattr(tk, "balance_sheet", pd.DataFrame())
    cf  = getattr(tk, "cashflow", pd.DataFrame())

    revenue = _row(inc, "Total Revenue", "TotalRevenue", "Revenue")
    ebit    = _row(inc, "EBIT", "Operating Income", "OperatingIncome")
    pretax  = _row(inc, "Pretax Income", "Income Before Tax")
    taxprov = _row(inc, "Tax Provision", "Income Tax Expense")

    da    = _row(cf, "Depreciation And Amortization",
                 "Depreciation Amortization Depletion", "Depreciation")
    capex = _row(cf, "Capital Expenditure", "Capital Expenditures", "Purchase Of PPE")

    total_debt = _row(bs, "Total Debt")
    if total_debt.dropna().empty:
        ltd = _row(bs, "Long Term Debt")
        std = _row(bs, "Current Debt", "Short Term Debt",
                   "Current Debt And Capital Lease Obligation")
        total_debt = ltd.add(std, fill_value=0)
    cash = _row(bs, "Cash And Cash Equivalents",
                "Cash Cash Equivalents And Short Term Investments",
                "Cash And Cash Equivalents And Short Term Investments")
    curr_assets = _row(bs, "Current Assets", "Total Current Assets")
    curr_liab   = _row(bs, "Current Liabilities", "Total Current Liabilities")

    # ---- histories (oldest -> newest), used by v2 driver estimation ----
    rev_h  = _hist(revenue)
    ebit_h = _hist(ebit)
    out["revenue_history"] = rev_h
    out["ebit_margin_history"] = (ebit_h / rev_h).dropna()
    out["nwc_history"] = (_hist(curr_assets) - _hist(curr_liab)).dropna()

    # ---- latest values ----
    out["revenue"]    = _latest(revenue)
    out["ebit"]       = _latest(ebit)
    out["pretax"]     = _latest(pretax)
    out["tax_prov"]   = _latest(taxprov)
    _da, _cx          = _latest(da), _latest(capex)
    out["da"]         = abs(_da) if not np.isnan(_da) else np.nan
    out["capex"]      = abs(_cx) if not np.isnan(_cx) else np.nan
    out["total_debt"] = _latest(total_debt)
    out["cash"]       = _latest(cash)
    cur_a, cur_l      = _latest(curr_assets), _latest(curr_liab)
    out["nwc"]        = (cur_a - cur_l) if not (np.isnan(cur_a) or np.isnan(cur_l)) else np.nan

    # ---- market data ----
    fast = {}
    try:
        fast = dict(tk.fast_info)
    except Exception:
        pass
    info = {}
    try:
        info = tk.info or {}
    except Exception:
        out["warnings"].append("tk.info unavailable")

    price = fast.get("last_price") or info.get("currentPrice") or info.get("regularMarketPrice")
    if price is None:
        try:
            price = float(tk.history(period="5d")["Close"].dropna().iloc[-1])
        except Exception:
            price = np.nan
    out["price"] = float(price) if price else np.nan

    shares = (fast.get("shares") or info.get("sharesOutstanding")
              or info.get("impliedSharesOutstanding"))
    out["shares"] = float(shares) if shares else np.nan

    beta_yahoo = info.get("beta")
    out["beta_yahoo"] = float(beta_yahoo) if beta_yahoo else np.nan

    out["name"]   = info.get("longName") or info.get("shortName") or ticker
    out["sector"] = info.get("sector", "n/a")
    out["interest_expense"] = abs(_latest(_row(inc, "Interest Expense",
                                                "Interest Expense Non Operating")))

    essential = ["revenue", "ebit", "price", "shares"]
    missing = [k for k in essential if np.isnan(out.get(k, np.nan))]
    if missing:
        out["ok"] = False
        out["warnings"].append(f"missing essential fields: {missing}")
    return out

try:
    F = fetch_fundamentals(CONFIG["TICKER"])
except Exception as e:
    F = {"ticker": CONFIG["TICKER"], "ok": False, "warnings": [f"fetch raised: {e}"]}

print(f"{F.get('name', CONFIG['TICKER'])}  ({F['ticker']}) | sector: {F.get('sector','n/a')}")
print("Pull OK:", F["ok"], "| warnings:", F["warnings"])
if not np.isnan(F.get("revenue", np.nan)):
    print(f"  Revenue (latest):   {F['revenue']/1e9:,.2f} B")
    print(f"  EBIT (latest):      {F['ebit']/1e9:,.2f} B")
    print(f"  Price / Shares:     {F.get('price', float('nan')):.2f} / "
          f"{F.get('shares', float('nan'))/1e9:,.3f} B")
    print(f"  Statement years available: {len(F.get('revenue_history', []))}")

### 3a. Manual fallback (only if the pull failed)

If the cell above says `Pull OK: False`, fill these in from the company's 10-K (absolute dollars, newest fiscal year; `revenue_history` oldest→newest) and re-run from here.

In [ ]:
# ============================================================================
# Cell 10 — Manual fallback (used only if F["ok"] is False)
# ============================================================================
MANUAL_FINANCIALS = {
    # "revenue": 383_285_000_000,
    # "ebit": 114_301_000_000,
    # "tax_rate": 0.15,
    # "da": 11_519_000_000,
    # "capex": 10_959_000_000,
    # "total_debt": 111_088_000_000,
    # "cash": 61_555_000_000,
    # "price": 190.0,
    # "shares": 15_500_000_000,
    # "beta": 1.25,                    # used only if OLS estimation also fails
    # "revenue_history": [274_515e6, 365_817e6, 394_328e6, 383_285e6],
}

if not F.get("ok", False) and MANUAL_FINANCIALS:
    print("Applying manual overrides...")
    for k, v in MANUAL_FINANCIALS.items():
        if k == "revenue_history":
            F[k] = pd.Series(v, dtype="float64")
        elif k == "tax_rate":
            F["_manual_tax_rate"] = v
        elif k == "beta":
            F["beta_yahoo"] = v
        else:
            F[k] = v
    F["ok"] = all(not np.isnan(F.get(x, np.nan)) for x in ["revenue", "ebit", "price", "shares"])
    print("Manual override applied. OK now:", F["ok"])
elif not F.get("ok", False):
    print("Pull failed AND no manual data supplied. Fill MANUAL_FINANCIALS above and re-run.")
else:
    print("Automatic pull was fine; manual fallback skipped.")

## 4. Derive drivers from history (v2 estimation)

Three upgrades over the naive version:

1. **Incremental NWC.** The FCF drag from working capital is driven by the *change* in NWC per dollar of *revenue change* — a marginal ratio — not by the NWC level. We estimate it as the median of ΔNWC/ΔRevenue across available statement years (clipped for sanity). If the history is too thin, the model falls back to the level proxy and says so out loud.
2. **Long-run margin.** The margin path converges to the multi-year average margin rather than assuming today's (possibly peak) margin holds forever.
3. **Historical CAGR by log-linear regression.** Instead of the endpoint-to-endpoint CAGR — which whipsaws with whichever two years the data source happens to return — we regress ln(revenue) on time and annualize the slope, and we label how many years the estimate is based on. Both figures are printed so you can see how much the method choice matters.

In [ ]:
# ============================================================================
# Cell 12 — Derive operating drivers (v2)
# ============================================================================
def _by_year(s: pd.Series) -> pd.Series:
    """Re-index a date-indexed history by fiscal year for cross-statement alignment."""
    s = s.dropna()
    try:
        return pd.Series(s.values, index=[ts.year for ts in s.index]).groupby(level=0).last()
    except Exception:
        return s

def incremental_nwc_ratio(F: Dict) -> Tuple[Optional[float], str]:
    """Median dNWC/dRevenue across years; None if not computable."""
    nwc_y = _by_year(F.get("nwc_history", pd.Series(dtype="float64")))
    rev_y = _by_year(F.get("revenue_history", pd.Series(dtype="float64")))
    df = pd.concat([nwc_y.rename("nwc"), rev_y.rename("rev")], axis=1).dropna()
    if len(df) < 2:
        return None, "insufficient history"
    d = df.diff().dropna()
    d = d[d["rev"].abs() > 0.001 * df["rev"].mean()]  # skip near-zero denominators
    if len(d) < 3:
        # with <3 observations a median of noisy ratios isn't trustworthy
        return None, f"only {len(d)} usable yearly change(s)"
    ratios = (d["nwc"] / d["rev"]).clip(-0.35, 0.35)
    return float(ratios.median()), f"median of {len(ratios)} yearly dNWC/dRev"

def loglinear_cagr(rev_hist: pd.Series) -> Tuple[Optional[float], int]:
    """CAGR from OLS of ln(revenue) on time. Returns (cagr, n_points)."""
    r = pd.to_numeric(rev_hist, errors="coerce").dropna()
    r = r[r > 0]
    if len(r) < 2:
        return None, len(r)
    slope = np.polyfit(np.arange(len(r)), np.log(r.values), 1)[0]
    return float(np.exp(slope) - 1.0), len(r)

def endpoint_cagr(rev_hist: pd.Series) -> Optional[float]:
    r = pd.to_numeric(rev_hist, errors="coerce").dropna()
    r = r[r > 0]
    if len(r) < 2:
        return None
    return float((r.iloc[-1] / r.iloc[0]) ** (1 / (len(r) - 1)) - 1)

def derive_drivers(F: Dict, cfg: Dict) -> Dict:
    rev = F["revenue"]
    d = {}

    # --- margins: starting level and long-run convergence target ---
    d["ebit_margin0"] = cfg["EBIT_MARGIN"] if cfg["EBIT_MARGIN"] is not None else \
        (F["ebit"] / rev if rev else np.nan)
    mh = F.get("ebit_margin_history", pd.Series(dtype="float64")).dropna()
    if cfg["LONG_RUN_EBIT_MARGIN"] is not None:
        d["ebit_margin_lr"], d["margin_lr_source"] = cfg["LONG_RUN_EBIT_MARGIN"], "config"
    elif len(mh) >= 2:
        d["ebit_margin_lr"] = float(mh.mean())
        d["margin_lr_source"] = f"mean of {len(mh)} yrs"
    else:
        d["ebit_margin_lr"], d["margin_lr_source"] = d["ebit_margin0"], "current (no history)"

    # --- tax rate ---
    if cfg["TAX_RATE"] is not None:
        d["tax_rate"] = cfg["TAX_RATE"]
    elif F.get("_manual_tax_rate") is not None:
        d["tax_rate"] = F["_manual_tax_rate"]
    elif not np.isnan(F.get("tax_prov", np.nan)) and not np.isnan(F.get("pretax", np.nan)) and F["pretax"]:
        d["tax_rate"] = float(np.clip(F["tax_prov"] / F["pretax"], 0.0, 0.45))
    else:
        d["tax_rate"] = 0.21

    # --- D&A / capex intensity ---
    d["da_pct"] = cfg["DA_PCT_REV"] if cfg["DA_PCT_REV"] is not None else \
        (F["da"] / rev if (rev and not np.isnan(F.get("da", np.nan))) else 0.03)
    d["capex_pct"] = cfg["CAPEX_PCT_REV"] if cfg["CAPEX_PCT_REV"] is not None else \
        (F["capex"] / rev if (rev and not np.isnan(F.get("capex", np.nan))) else 0.04)

    # --- incremental NWC (v2 fix) ---
    if cfg["NWC_INCREMENTAL_PCT"] is not None:
        d["nwc_inc_pct"], d["nwc_method"] = cfg["NWC_INCREMENTAL_PCT"], "config"
    else:
        inc, how = incremental_nwc_ratio(F)
        if inc is not None:
            d["nwc_inc_pct"], d["nwc_method"] = inc, how
        elif not np.isnan(F.get("nwc", np.nan)) and rev:
            d["nwc_inc_pct"] = float(np.clip(F["nwc"] / rev, -0.30, 0.40))
            d["nwc_method"] = "FALLBACK level proxy (NWC/rev)"
        else:
            d["nwc_inc_pct"], d["nwc_method"] = 0.0, "FALLBACK zero (no data)"

    # --- historical growth (v2: regression-based, window labeled) ---
    rh = F.get("revenue_history", pd.Series(dtype="float64"))
    cagr_ll, n_pts = loglinear_cagr(rh)
    cagr_ep = endpoint_cagr(rh)
    d["hist_rev_cagr"] = cagr_ll if cagr_ll is not None else (cagr_ep if cagr_ep is not None else np.nan)
    d["hist_cagr_endpoint"] = cagr_ep if cagr_ep is not None else np.nan
    d["cagr_window"] = max(n_pts - 1, 0)
    return d

DRV = derive_drivers(F, CONFIG)

rows = [
    ("EBIT margin (start)",  f"{DRV['ebit_margin0']:.2%}"),
    ("EBIT margin (long-run)", f"{DRV['ebit_margin_lr']:.2%}  [{DRV['margin_lr_source']}]"),
    ("Tax rate",             f"{DRV['tax_rate']:.2%}"),
    ("D&A % rev",            f"{DRV['da_pct']:.2%}"),
    ("Capex % rev",          f"{DRV['capex_pct']:.2%}"),
    ("Incremental NWC (dNWC/dRev)", f"{DRV['nwc_inc_pct']:.2%}  [{DRV['nwc_method']}]"),
    (f"Hist. revenue CAGR (log-linear, {DRV['cagr_window']}y window)",
     f"{DRV['hist_rev_cagr']:.2%}" if pd.notna(DRV['hist_rev_cagr']) else "n/a"),
    ("Hist. revenue CAGR (endpoint, same window)",
     f"{DRV['hist_cagr_endpoint']:.2%}" if pd.notna(DRV['hist_cagr_endpoint']) else "n/a"),
]
print(pd.DataFrame(rows, columns=["Driver", "Value"]).to_string(index=False))
if DRV["cagr_window"] < 5:
    print(f"\n  caveat: CAGR is estimated from only {DRV['cagr_window']} year(s) of change — "
          "treat the expectations-gap comparison accordingly.")

## 5. Beta by OLS (v2)

Instead of accepting Yahoo's single opaque beta, estimate it directly: regress the stock's **weekly** returns on SPY's weekly returns over the lookback window. The OLS slope is beta; R² tells you how much of the stock's variance the market explains. Yahoo's figure is kept as a cross-check and fallback. This is the same regression machinery as any econometrics coursework — β = Cov(rₛ, rₘ)/Var(rₘ) is literally the univariate OLS slope.

In [ ]:
# ============================================================================
# Cell 14 — Estimate beta via OLS of weekly returns vs SPY
# ============================================================================
def estimate_beta_ols(ticker: str, benchmark: str = "SPY",
                      years: int = 5) -> Optional[Dict]:
    """OLS beta from weekly returns. Returns dict or None on failure."""
    try:
        raw = yf.download([ticker, benchmark], period=f"{years}y", interval="1wk",
                          auto_adjust=True, progress=False)
        closes = raw["Close"] if "Close" in raw else raw
        if isinstance(closes, pd.Series) or ticker not in closes.columns \
           or benchmark not in closes.columns:
            return None
        rets = closes[[ticker, benchmark]].dropna().pct_change().dropna()
        if len(rets) < 52:                      # need at least ~1y of weeks
            return None
        rs, rm = rets[ticker], rets[benchmark]
        var_m = rm.var()
        if var_m <= 0:
            return None
        beta = float(rs.cov(rm) / var_m)        # univariate OLS slope
        r2 = float(rs.corr(rm) ** 2)
        return {"beta": beta, "r2": r2, "n_weeks": int(len(rets))}
    except Exception as e:
        print("  OLS beta estimation failed:", e)
        return None

BETA_OLS = estimate_beta_ols(CONFIG["TICKER"], years=CONFIG["BETA_LOOKBACK_YEARS"])
beta_yahoo = F.get("beta_yahoo", np.nan)

if CONFIG["BETA_OVERRIDE"] is not None:
    beta_used, beta_src = float(CONFIG["BETA_OVERRIDE"]), "config override"
elif BETA_OLS is not None and 0.2 < BETA_OLS["beta"] < 3.0:
    beta_used = BETA_OLS["beta"]
    beta_src = (f"OLS, weekly vs SPY, {BETA_OLS['n_weeks']} obs, "
                f"R^2={BETA_OLS['r2']:.2f}")
elif not np.isnan(beta_yahoo):
    beta_used, beta_src = float(beta_yahoo), "Yahoo (OLS unavailable)"
else:
    beta_used, beta_src = 1.0, "neutral default"

F["beta_used"], F["beta_source"] = beta_used, beta_src

print(f"OLS beta:   {BETA_OLS['beta']:.3f}  (R^2={BETA_OLS['r2']:.2f}, "
      f"n={BETA_OLS['n_weeks']} weeks)" if BETA_OLS else "OLS beta:   unavailable")
print(f"Yahoo beta: {beta_yahoo:.3f}" if not np.isnan(beta_yahoo) else "Yahoo beta: unavailable")
print(f"USED:       {beta_used:.3f}  [{beta_src}]")

## 6. WACC build (CAPM)

Cost of equity via CAPM ($r_f + \beta \cdot \text{ERP}$) using the beta chosen above; after-tax cost of debt inferred from interest expense where sane; weights by market equity vs book debt. The `[MIN_WACC, MAX_WACC]` guardrails still apply, and the cell announces loudly if they bind — because a clipped WACC silently changes what the reverse-DCF's implied growth means.

In [ ]:
# ============================================================================
# Cell 16 — WACC
# ============================================================================
def compute_wacc(F: Dict, DRV: Dict, cfg: Dict, risk_free: float) -> Dict:
    beta = F.get("beta_used", 1.0)
    erp = cfg["EQUITY_RISK_PREMIUM"]
    cost_equity = risk_free + beta * erp

    cod_pre = cfg["PRETAX_COST_OF_DEBT"]
    ie, td = F.get("interest_expense", np.nan), F.get("total_debt", np.nan)
    if not np.isnan(ie) and not np.isnan(td) and td > 0:
        implied = ie / td
        if 0.005 < implied < 0.20:
            cod_pre = implied
    cost_debt_at = cod_pre * (1 - DRV["tax_rate"])

    e_val = F.get("price", np.nan) * F.get("shares", np.nan)
    d_val = F.get("total_debt", np.nan)
    e_val = 0.0 if np.isnan(e_val) else e_val
    d_val = 0.0 if np.isnan(d_val) else d_val
    v = e_val + d_val
    we = e_val / v if v else 1.0
    wd = d_val / v if v else 0.0

    wacc_raw = we * cost_equity + wd * cost_debt_at
    wacc = float(np.clip(wacc_raw, cfg["MIN_WACC"], cfg["MAX_WACC"]))
    return {"beta": beta, "erp": erp, "cost_equity": cost_equity,
            "cost_debt_pretax": cod_pre, "cost_debt_aftertax": cost_debt_at,
            "weight_equity": we, "weight_debt": wd,
            "wacc_raw": wacc_raw, "wacc": wacc}

W = compute_wacc(F, DRV, CONFIG, risk_free)

wacc_tbl = pd.DataFrame({
    "Component": ["Risk-free", "Beta (used)", "Equity risk premium", "Cost of equity",
                  "Pre-tax cost of debt", "After-tax cost of debt",
                  "Weight equity", "Weight debt", "WACC (raw)", "WACC (used)"],
    "Value": [f"{risk_free:.2%}", f"{W['beta']:.3f}", f"{W['erp']:.2%}",
              f"{W['cost_equity']:.2%}", f"{W['cost_debt_pretax']:.2%}",
              f"{W['cost_debt_aftertax']:.2%}", f"{W['weight_equity']:.2%}",
              f"{W['weight_debt']:.2%}", f"{W['wacc_raw']:.2%}", f"{W['wacc']:.2%}"],
})
print(wacc_tbl.to_string(index=False))
if abs(W["wacc_raw"] - W["wacc"]) > 1e-9:
    print(f"\n  WARNING: raw WACC {W['wacc_raw']:.2%} clipped to {W['wacc']:.2%} — "
          "implied-growth results now partly reflect this clip, not pure market pricing.")

## 7. The core engine: `DCFModel` v2

Same design as before — one object, two query directions — with the v2 economics inside:

- **`growth_path(g_start)`**: revenue growth fades **linearly** from `g_start` in year 1 to `terminal_growth` in the final explicit year, so the handoff into the Gordon terminal value has no growth discontinuity.
- **`margin_path()`**: EBIT margin drifts linearly from the current margin to the long-run level over the horizon.
- **`horizon_cagr(g_start)`**: the compound growth the whole fading path implies — this, not the initial rate, is what's comparable to a historical CAGR.
- **`implied_growth(price)`**: the reverse direction — Brent's method finds the `g_start` whose valuation equals the market price. It works because value is monotonic in `g_start` (every year's growth rises with it), which the validation cell verifies numerically rather than taking on faith.

In [ ]:
# ============================================================================
# Cell 18 — DCFModel v2 (shared forward/reverse engine, fading growth)
# ============================================================================
@dataclass
class DCFModel:
    revenue0: float
    shares: float
    net_debt: float
    wacc: float
    ebit_margin0: float
    ebit_margin_lr: float
    tax_rate: float
    da_pct: float
    capex_pct: float
    nwc_inc_pct: float
    years: int = 10
    terminal_growth: float = 0.025
    exit_ev_ebitda: float = 14.0

    # ---- v2 paths ----
    def growth_path(self, g_start: float) -> np.ndarray:
        """Linear fade: year-1 growth = g_start, final-year growth = terminal."""
        n = self.years
        if n == 1:
            return np.array([self.terminal_growth])
        w = np.arange(n) / (n - 1)                     # 0 -> 1
        return (1 - w) * g_start + w * self.terminal_growth

    def margin_path(self) -> np.ndarray:
        """Linear drift from current margin to long-run margin over the horizon."""
        return np.linspace(self.ebit_margin0, self.ebit_margin_lr, self.years + 1)[1:]

    def horizon_cagr(self, g_start: float) -> float:
        """Compound annual growth implied by the whole fading path."""
        factor = float(np.prod(1.0 + self.growth_path(g_start)))
        return factor ** (1.0 / self.years) - 1.0

    # ---- projection ----
    def project_fcf(self, g_start: float) -> pd.DataFrame:
        growths, margins = self.growth_path(g_start), self.margin_path()
        rows, rev_prev = [], self.revenue0
        for yr in range(1, self.years + 1):
            g, m = growths[yr - 1], margins[yr - 1]
            rev = rev_prev * (1 + g)
            ebit = rev * m
            nopat = ebit * (1 - self.tax_rate)
            da = rev * self.da_pct
            capex = rev * self.capex_pct
            d_nwc = (rev - rev_prev) * self.nwc_inc_pct   # incremental NWC (v2 fix)
            fcf = nopat + da - capex - d_nwc
            disc = (1 + self.wacc) ** yr
            rows.append({"Year": yr, "Growth": g, "Revenue": rev, "Margin": m,
                         "EBIT": ebit, "NOPAT": nopat, "D&A": da, "Capex": capex,
                         "dNWC": d_nwc, "FCF": fcf,
                         "DiscFactor": 1 / disc, "PV_FCF": fcf / disc})
            rev_prev = rev
        return pd.DataFrame(rows)

    def terminal_value(self, sched: pd.DataFrame, method: str = "gordon") -> float:
        last = sched.iloc[-1]
        if method == "gordon":
            denom = self.wacc - self.terminal_growth
            if denom <= 0:
                return np.nan
            # final explicit year already grows at terminal rate -> seamless handoff
            return last["FCF"] * (1 + self.terminal_growth) / denom
        elif method == "exit":
            return self.exit_ev_ebitda * (last["EBIT"] + last["D&A"])  # EBITDA proxy
        raise ValueError("method must be 'gordon' or 'exit'")

    def enterprise_value(self, g_start: float,
                         tv_method: str = "gordon") -> Tuple[float, pd.DataFrame, float]:
        sched = self.project_fcf(g_start)
        tv = self.terminal_value(sched, tv_method)
        pv_tv = tv / ((1 + self.wacc) ** self.years) if not np.isnan(tv) else np.nan
        ev = sched["PV_FCF"].sum() + (pv_tv if not np.isnan(pv_tv) else 0.0)
        return ev, sched, pv_tv

    def intrinsic_value_per_share(self, g_start: float, tv_method: str = "gordon") -> float:
        ev, _, _ = self.enterprise_value(g_start, tv_method)
        if self.shares <= 0 or np.isnan(self.shares):
            return np.nan
        return (ev - self.net_debt) / self.shares

    # ---- reverse direction ----
    def implied_growth(self, target_price: float, tv_method: str = "gordon",
                       lo: float = -0.20, hi: float = 0.80) -> Optional[float]:
        """Solve for the g_start whose valuation equals target_price."""
        f = lambda g: self.intrinsic_value_per_share(g, tv_method) - target_price
        try:
            f_lo, f_hi = f(lo), f(hi)
            if np.isnan(f_lo) or np.isnan(f_hi) or f_lo * f_hi > 0:
                return None            # price not reachable within the bracket
            return float(optimize.brentq(f, lo, hi, xtol=1e-6, maxiter=200))
        except Exception:
            return None

# ---- instantiate from assembled data ----
net_debt = (0.0 if np.isnan(F.get("total_debt", np.nan)) else F["total_debt"]) \
         - (0.0 if np.isnan(F.get("cash", np.nan)) else F["cash"])

model = DCFModel(
    revenue0=F["revenue"], shares=F["shares"], net_debt=net_debt, wacc=W["wacc"],
    ebit_margin0=DRV["ebit_margin0"], ebit_margin_lr=DRV["ebit_margin_lr"],
    tax_rate=DRV["tax_rate"], da_pct=DRV["da_pct"], capex_pct=DRV["capex_pct"],
    nwc_inc_pct=DRV["nwc_inc_pct"],
    years=CONFIG["PROJECTION_YEARS"], terminal_growth=CONFIG["TERMINAL_GROWTH"],
    exit_ev_ebitda=CONFIG["EXIT_EV_EBITDA"],
)

print("DCFModel v2 ready.")
print(f"  revenue0={model.revenue0/1e9:,.2f}B | shares={model.shares/1e9:,.3f}B | "
      f"net_debt={model.net_debt/1e9:,.2f}B | WACC={model.wacc:.2%}")
print(f"  growth: {CONFIG['BASE_INITIAL_GROWTH']:.1%} fading to "
      f"{model.terminal_growth:.1%} over {model.years}y | "
      f"margin: {model.ebit_margin0:.1%} -> {model.ebit_margin_lr:.1%}")

## 8. Validation — the model checks itself

Two properties get verified on every run, with real numbers rather than assumptions:

1. **Round-trip:** value the company forward at the base growth, feed that price into the reverse solver, and confirm it recovers the original growth to within tolerance. This is the proof that forward and reverse are genuinely one model.
2. **Monotonicity:** sample the value function across the whole solver bracket and confirm it's strictly increasing in `g_start`. Monotonicity is what guarantees the implied growth is *unique* — without it, "the" market-implied growth wouldn't be a well-defined number.

If either check fails, the printout says so before you read anything downstream.

In [ ]:
# ============================================================================
# Cell 20 — Self-tests: round-trip + monotonicity + structural guards
# ============================================================================
lo, hi = CONFIG["IMPLIED_GROWTH_MIN"], CONFIG["IMPLIED_GROWTH_MAX"]
all_ok = True

# 1) forward -> reverse round-trip
g_in = CONFIG["BASE_INITIAL_GROWTH"]
v_fwd = model.intrinsic_value_per_share(g_in, "gordon")
g_back = model.implied_growth(v_fwd, "gordon", lo=lo, hi=hi)
if g_back is not None and abs(g_back - g_in) < 1e-4:
    print(f"[PASS] Round-trip: forward({g_in:.2%}) -> ${v_fwd:,.2f} -> "
          f"reverse -> {g_back:.4%}")
else:
    all_ok = False
    print(f"[FAIL] Round-trip: got {g_back} (expected ~{g_in:.2%})")

# 2) monotonicity of value in g_start across the solver bracket
g_grid = np.linspace(lo, hi, 25)
v_grid = np.array([model.intrinsic_value_per_share(g, "gordon") for g in g_grid])
diffs = np.diff(v_grid)
if np.all(diffs > 0):
    print(f"[PASS] Monotonic: value strictly increasing in growth over "
          f"[{lo:.0%}, {hi:.0%}] (min step ${diffs.min():,.2f})")
else:
    all_ok = False
    print("[FAIL] Value NOT monotonic in growth — implied growth may be non-unique. "
          "Check nwc_inc_pct / margin inputs.")

# 3) structural guards
gp = model.growth_path(g_in)
assert abs(gp[0] - g_in) < 1e-12 and abs(gp[-1] - model.terminal_growth) < 1e-12, \
    "growth path endpoints wrong"
print(f"[PASS] Growth path endpoints: year1={gp[0]:.2%}, year{model.years}={gp[-1]:.2%}")

if model.terminal_growth < model.wacc:
    print(f"[PASS] Gordon condition: terminal growth {model.terminal_growth:.2%} "
          f"< WACC {model.wacc:.2%}")
else:
    all_ok = False
    print("[FAIL] Terminal growth >= WACC — Gordon terminal value undefined.")

print("\nAll validation checks passed." if all_ok
      else "\nVALIDATION ISSUES — fix before trusting downstream results.")

## 9. FORWARD valuation — base / bull / bear

Scenarios shift the **initial** growth rate; every path still fades to the same terminal rate. The full base-case schedule shows the per-year growth and margin so the fade is auditable, and the terminal value's share of enterprise value is printed — with a 10-year horizon it should carry noticeably less of the total than a 5-year model would.

In [ ]:
# ============================================================================
# Cell 22 — Forward valuation across scenarios
# ============================================================================
base_g = CONFIG["BASE_INITIAL_GROWTH"]
scenarios = {"Bear": base_g - CONFIG["BEAR_DELTA"],
             "Base": base_g,
             "Bull": base_g + CONFIG["BULL_DELTA"]}

rows = []
for name, g in scenarios.items():
    for tv in ["gordon", "exit"]:
        rows.append({"Scenario": name, "Initial growth": g,
                     "Horizon CAGR": model.horizon_cagr(g), "TV method": tv,
                     "Value/Share": model.intrinsic_value_per_share(g, tv)})
fwd = pd.DataFrame(rows)

price = F.get("price", np.nan)
fwd["Upside vs price"] = fwd["Value/Share"] / price - 1 if pd.notna(price) else np.nan

disp = fwd.copy()
disp["Initial growth"] = disp["Initial growth"].map("{:.1%}".format)
disp["Horizon CAGR"] = disp["Horizon CAGR"].map("{:.1%}".format)
disp["Value/Share"] = disp["Value/Share"].map(lambda x: f"${x:,.2f}" if pd.notna(x) else "n/a")
disp["Upside vs price"] = disp["Upside vs price"].map(lambda x: f"{x:+.1%}" if pd.notna(x) else "n/a")
print(f"Current market price: ${price:,.2f}\n" if pd.notna(price) else "Current price unavailable\n")
print(disp.to_string(index=False))

ev_base, sched_base, pv_tv_base = model.enterprise_value(base_g, "gordon")
show = sched_base.copy()
show["Growth"] = sched_base["Growth"].map("{:.1%}".format)
show["Margin"] = sched_base["Margin"].map("{:.1%}".format)
for c in ["Revenue", "EBIT", "NOPAT", "D&A", "Capex", "dNWC", "FCF", "PV_FCF"]:
    show[c] = sched_base[c].map(lambda x: f"{x/1e9:,.1f}B")
show["DiscFactor"] = sched_base["DiscFactor"].map("{:.3f}".format)
print("\nBase-case unlevered FCF schedule (Gordon TV):")
print(show.to_string(index=False))
print(f"\n  PV of explicit FCF:   {sched_base['PV_FCF'].sum()/1e9:,.2f}B")
print(f"  PV of terminal value: {pv_tv_base/1e9:,.2f}B  ({pv_tv_base/ev_base:.0%} of EV)")
print(f"  Enterprise value:     {ev_base/1e9:,.2f}B")
print(f"  Equity value:         {(ev_base - model.net_debt)/1e9:,.2f}B")

### 9a. Sensitivity grid — WACC × terminal growth

The classic two-way table over the two assumptions that dominate terminal value. Each cell is intrinsic value per share at the base fading-growth path.

In [ ]:
# ============================================================================
# Cell 24 — Sensitivity heatmap (WACC x terminal growth)
# ============================================================================
wacc_axis = np.round(np.linspace(max(CONFIG["MIN_WACC"], model.wacc - 0.03),
                                 min(CONFIG["MAX_WACC"], model.wacc + 0.03), 7), 4)
tg_axis = np.round(np.linspace(max(0.0, model.terminal_growth - 0.015),
                               model.terminal_growth + 0.015, 7), 4)

grid = np.full((len(tg_axis), len(wacc_axis)), np.nan)
for i, tg in enumerate(tg_axis):
    for j, wc in enumerate(wacc_axis):
        m = DCFModel(**{**model.__dict__, "wacc": float(wc), "terminal_growth": float(tg)})
        grid[i, j] = m.intrinsic_value_per_share(base_g, "gordon")

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(grid, cmap="RdYlGn", aspect="auto", origin="lower")
ax.set_xticks(range(len(wacc_axis)))
ax.set_xticklabels([f"{w:.2%}" for w in wacc_axis], rotation=45, ha="right")
ax.set_yticks(range(len(tg_axis)))
ax.set_yticklabels([f"{t:.2%}" for t in tg_axis])
ax.set_xlabel("WACC"); ax.set_ylabel("Terminal growth")
ax.set_title(f"{F['ticker']} — Intrinsic Value / Share Sensitivity (fading-growth base case)")
for i in range(len(tg_axis)):
    for j in range(len(wacc_axis)):
        if not np.isnan(grid[i, j]):
            ax.text(j, i, f"${grid[i, j]:,.0f}", ha="center", va="center", fontsize=8)
cbar = fig.colorbar(im, ax=ax); cbar.set_label("Value / Share ($)")
plt.tight_layout(); plt.show()

if pd.notna(price):
    print(f"Reference: current price ${price:,.2f}. Green = intrinsic value above price.")

## 10. REVERSE valuation — what is the market pricing in?

Hold everything fixed except growth; hold the market price fixed; solve for the growth path that reconciles them. Two numbers come out, and the distinction matters:

- **Implied initial growth** — the year-1 rate the fading path must start at. This is the solver's native output.
- **Implied horizon CAGR** — the compound growth over the whole 10-year path. **This is the number to compare against historical CAGR**, because both are multi-year compound rates. Comparing the initial rate to a historical CAGR would overstate the gap.

A small WACC-sensitivity table shows how much the implied growth moves if the discount rate is ±1% — the honest acknowledgment that "what the market is pricing in" depends on the discount rate you assume it's using.

In [ ]:
# ============================================================================
# Cell 26 — Reverse DCF: implied growth + WACC sensitivity
# ============================================================================
lo, hi = CONFIG["IMPLIED_GROWTH_MIN"], CONFIG["IMPLIED_GROWTH_MAX"]
implied = {}
if pd.notna(price):
    for tv in ["gordon", "exit"]:
        implied[tv] = model.implied_growth(price, tv, lo=lo, hi=hi)
else:
    print("No market price — reverse DCF skipped.")

hist_cagr = DRV.get("hist_rev_cagr", np.nan)

if implied:
    print(f"Market price: ${price:,.2f}\n")
    for tv, g0 in implied.items():
        if g0 is None:
            print(f"  [{tv:>6}] not solvable in [{lo:.0%}, {hi:.0%}] — price outside "
                  "the model's reachable range under these assumptions")
        else:
            print(f"  [{tv:>6}] implied initial growth: {g0:.2%}   "
                  f"-> implied {model.years}y horizon CAGR: {model.horizon_cagr(g0):.2%}")

    g0_ref = implied.get("gordon")
    if g0_ref is not None and pd.notna(hist_cagr):
        icagr = model.horizon_cagr(g0_ref)
        gap = icagr - hist_cagr
        print(f"\n  Historical revenue CAGR (log-linear, {DRV['cagr_window']}y): {hist_cagr:.2%}")
        print(f"  Expectations gap (implied horizon CAGR - historical):  {gap:+.2%}")
        verdict = ("market is pricing in growth ABOVE its recent history (optimistic)"
                   if gap > 0.01 else
                   "market is pricing in growth BELOW its recent history (cautious)"
                   if gap < -0.01 else
                   "market is pricing in growth roughly IN LINE with its history")
        print(f"  Read: {verdict}.")

    # WACC sensitivity of the implied growth (gordon)
    print("\n  Implied growth vs WACC (gordon):")
    for dw in (-0.01, 0.0, 0.01):
        wc = float(np.clip(model.wacc + dw, CONFIG["MIN_WACC"], CONFIG["MAX_WACC"]))
        m = DCFModel(**{**model.__dict__, "wacc": wc})
        g0w = m.implied_growth(price, "gordon", lo=lo, hi=hi)
        if g0w is None:
            print(f"    WACC {wc:.2%}: not solvable in bracket")
        else:
            print(f"    WACC {wc:.2%}: initial {g0w:.2%} | horizon CAGR {m.horizon_cagr(g0w):.2%}")

### 10a. Price → implied-growth curve

Sweep prices from 50% to 160% of current and plot the **implied horizon CAGR** each price requires — same units as the historical-CAGR reference line, so the vertical distance between the red marker and the green line *is* the expectations gap.

In [ ]:
# ============================================================================
# Cell 28 — Price vs implied horizon-CAGR curve
# ============================================================================
if pd.notna(price):
    price_grid = np.linspace(price * 0.5, price * 1.6, 40)
    pts = []
    for p in price_grid:
        g0 = model.implied_growth(p, "gordon", lo=lo, hi=hi)
        if g0 is not None:
            pts.append((p, model.horizon_cagr(g0)))

    if pts:
        xs, ys = zip(*pts)
        fig, ax = plt.subplots()
        ax.plot(xs, ys, lw=2.2, color="#2c3e50",
                label=f"Implied {model.years}y CAGR vs price")
        ax.axvline(price, color="#e74c3c", ls="--", lw=1.6,
                   label=f"Current price ${price:,.0f}")
        if pd.notna(hist_cagr):
            ax.axhline(hist_cagr, color="#27ae60", ls=":", lw=1.8,
                       label=f"Historical CAGR {hist_cagr:.1%}")
        g_now = model.implied_growth(price, "gordon", lo=lo, hi=hi)
        if g_now is not None:
            c_now = model.horizon_cagr(g_now)
            ax.scatter([price], [c_now], color="#e74c3c", zorder=5, s=60)
            ax.annotate(f"  market implies {c_now:.1%}/yr", (price, c_now),
                        fontsize=10, va="bottom")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
        ax.set_xlabel("Price / Share")
        ax.set_ylabel(f"Implied {model.years}y revenue CAGR")
        ax.set_title(f"{F['ticker']} — Reverse DCF: Price -> Implied Growth")
        ax.legend(frameon=False)
        plt.tight_layout(); plt.show()
    else:
        print("Curve empty: no price in the swept range is reachable within the growth bracket.")

## 11. Synthesis — *your view* vs *the market's view*

Left: the football field of forward values (bear/base/bull ranges across both terminal-value methods) against the market price. Right: implied horizon CAGR vs historical CAGR vs your base-case horizon CAGR — three compound growth rates in the same units, which is what makes the comparison legitimate.

In [ ]:
# ============================================================================
# Cell 30 — Synthesis: football field + expectations verdict
# ============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6),
                               gridspec_kw={"width_ratios": [1.3, 1]})

# ---- (left) football field ----
labels, lows, highs, mids = [], [], [], []
for name in ["Bear", "Base", "Bull"]:
    sub = fwd[fwd["Scenario"] == name]["Value/Share"].dropna()
    if len(sub):
        labels.append(name)
        lows.append(sub.min()); highs.append(sub.max()); mids.append(sub.mean())

ypos = np.arange(len(labels))
for i in range(len(labels)):
    ax1.barh(ypos[i], highs[i] - lows[i], left=lows[i], height=0.5,
             color="#5DADE2", edgecolor="#2874A6", alpha=0.85)
    ax1.text(mids[i], ypos[i], f"${mids[i]:,.0f}", va="center", ha="center",
             fontsize=10, fontweight="bold", color="#1B2631")
ax1.set_yticks(ypos); ax1.set_yticklabels(labels)
if pd.notna(price):
    ax1.axvline(price, color="#e74c3c", ls="--", lw=2, label=f"Market ${price:,.0f}")
    ax1.legend(frameon=False, loc="lower right")
ax1.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax1.set_xlabel("Intrinsic value / share")
ax1.set_title("Your View (Forward DCF) — value range vs market")

# ---- (right) growth comparison, all in horizon-CAGR units ----
g0_imp = implied.get("gordon") if implied else None
bars = []
if g0_imp is not None:
    bars.append(("Market-implied\nhorizon CAGR", model.horizon_cagr(g0_imp), "#e74c3c"))
if pd.notna(hist_cagr):
    bars.append((f"Historical CAGR\n({DRV['cagr_window']}y, log-linear)", hist_cagr, "#27ae60"))
bars.append(("Base-case\nhorizon CAGR", model.horizon_cagr(base_g), "#5DADE2"))

ax2.bar([b[0] for b in bars], [b[1] for b in bars],
        color=[b[2] for b in bars], alpha=0.85, edgecolor="black", linewidth=0.6)
for i, (_, v, _) in enumerate(bars):
    ax2.text(i, v + 0.003, f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax2.set_title("The Market's View (Reverse DCF)")
ax2.set_ylabel(f"{model.years}y compound growth rate")
ax2.axhline(0, color="black", lw=0.8)
plt.tight_layout(); plt.show()

# ---- verdict box ----
print("=" * 66)
print(f"  VALUATION SUMMARY — {F.get('name', F['ticker'])} ({F['ticker']})")
print("=" * 66)
base_val = fwd[(fwd.Scenario == "Base") & (fwd["TV method"] == "gordon")]["Value/Share"].iloc[0]
if pd.notna(price):
    print(f"  Current price:                    ${price:,.2f}")
print(f"  Base-case intrinsic value:        ${base_val:,.2f}  "
      f"(initial {base_g:.0%} fading to {model.terminal_growth:.1%})")
if pd.notna(price):
    up = base_val / price - 1
    print(f"  Forward signal:                   {up:+.1%} "
          f"({'UNDERvalued' if up > 0 else 'OVERvalued'} vs price, base case)")
if g0_imp is not None:
    print(f"  Market-implied initial growth:    {g0_imp:.2%}")
    print(f"  Market-implied horizon CAGR:      {model.horizon_cagr(g0_imp):.2%}")
if g0_imp is not None and pd.notna(hist_cagr):
    gap = model.horizon_cagr(g0_imp) - hist_cagr
    print(f"  Historical revenue CAGR:          {hist_cagr:.2%}  "
          f"({DRV['cagr_window']}y window)")
    print(f"  Expectations gap:                 {gap:+.2%} "
          f"({'market optimistic' if gap > 0 else 'market cautious'})")
print("=" * 66)
print("  The debate: is the market's implied growth path achievable?")
print("  Forward says what YOU think it's worth; reverse says what the")
print("  MARKET must believe. The gap between them is the investment case.")

## 12. What this notebook does — and a resume bullet

**Summary.** One `DCFModel` answers two questions. Forward: given a growth path that starts at an assumed initial rate and fades linearly to a terminal rate over ten years — with EBIT margin converging to its long-run historical level, incremental working capital charged against revenue *growth*, an OLS-estimated beta feeding a CAPM WACC, and dual terminal-value methods — what is the business worth per share? Reverse: given the market price, what growth path must the market believe? The reverse solver's output is reported both as an implied initial growth and as an implied horizon CAGR, the latter benchmarked against a log-linear historical CAGR with its estimation window labeled. The notebook validates itself on every run: a forward→reverse round-trip must recover the input growth, and the value function must be numerically monotonic across the solver bracket — the property that makes "the market-implied growth" a unique, well-defined number.

**Resume-ready bullet:**

> *Designed a two-stage DCF valuation engine in Python (forward and reverse): estimated equity beta via OLS regression of weekly returns against the S&P 500, modeled linearly fading revenue growth with margin convergence and incremental working-capital investment, and priced with dual terminal-value methods; inverted the same engine with a bracketed root-finder to recover the market-implied growth path from the share price, with automated round-trip and monotonicity validation on every run.*

---
*Honest caveats:* Yahoo statement history is short (often 4 years), so the historical CAGR — even regression-based — is a narrow-window estimate; the printout labels the window for exactly that reason. The incremental-NWC ratio estimated from so few observations is noisy and falls back to a level proxy (labeled) when history is insufficient. The implied growth is conditional on the WACC and margin assumptions being the market's too — the WACC-sensitivity table exists to size that dependence. And a linear fade is one choice of path shape; the implied *initial* rate would differ under other shapes even when the implied horizon CAGR is similar, which is why the horizon CAGR is the headline number.